# Day 01: The Scalar

**Series:** PyTorch from Scratch: The Zero-to-One Framework

This is the finished Day 01 lesson notebook. The code is written out so you can read it directly while recording.

## Goal

Build a scalar `Value` object that can participate in a computation graph and carry gradients.

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op="", label=""):
        self.data = float(data)
        self.grad = 0.0
        self._prev = set(_children)
        self._op = _op
        self.label = label
        self._backward = lambda: None

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out


## Compute A Tiny Graph

We want to evaluate:

$$d = a * b + c$$

In [ ]:
a = Value(2.0, label="a")
b = Value(-3.0, label="b")
c = Value(10.0, label="c")

e = a * b
e.label = "e"
d = e + c
d.label = "d"

print(d)
print(e)

## Local Derivatives

The only rules we need here are:

$$\frac{\partial (x + y)}{\partial x} = 1, \quad \frac{\partial (x + y)}{\partial y} = 1$$

$$\frac{\partial (x \cdot y)}{\partial x} = y, \quad \frac{\partial (x \cdot y)}{\partial y} = x$$

In [ ]:
d.grad = 1.0
d._backward()
e._backward()

print("dd/da =", a.grad)
print("dd/db =", b.grad)
print("dd/dc =", c.grad)
print("dd/de =", e.grad)

## What We Learned

A scalar is just a number with graph history and a place to store gradients. Day 02 automates the traversal, but the local derivative rules stay the same.